# 🚗 Vehicle Detection - Optimized Training
## Target: Maximum Accuracy

**Optimizations:**
- ✅ Pretrained weights (ImageNet)
- ✅ MobileNetV3-Large (bigger model)
- ✅ Advanced augmentation (Mixup, CutMix)
- ✅ Label smoothing
- ✅ Cosine annealing LR
- ✅ 50 epochs
- ✅ Test-Time Augmentation (TTA)

**⚠️ IMPORTANT:** Ve a **Settings > Internet > ON** antes de ejecutar

In [ ]:
# Setup
!pip install -q tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
from pathlib import Path
from PIL import Image, ImageEnhance, ImageFilter
from tqdm import tqdm
import json
import time
import random
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# Advanced Dataset with augmentation
class VehicleDataset(Dataset):
    def __init__(self, root, transform=None, mixup_alpha=0.2, use_mixup=True):
        self.images = []
        self.labels = []
        self.class_to_idx = {}
        self.transform = transform
        self.mixup_alpha = mixup_alpha
        self.use_mixup = use_mixup
        
        root = Path(root)
        for idx, class_dir in enumerate(sorted([d for d in root.iterdir() if d.is_dir()])):
            self.class_to_idx[class_dir.name] = idx
            for img in class_dir.glob('*.jpg'):
                self.images.append(img)
                self.labels.append(idx)
        
        self.idx_to_class = {v: k for k, v in self.class_to_idx.items()}
        self.num_classes = len(self.class_to_idx)
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img = Image.open(self.images[idx]).convert('RGB')
        label = self.labels[idx]
        
        if self.transform:
            img = self.transform(img)
        
        # Mixup augmentation
        if self.use_mixup and random.random() < 0.5:
            mix_idx = random.randint(0, len(self.images) - 1)
            mix_img = Image.open(self.images[mix_idx]).convert('RGB')
            mix_label = self.labels[mix_idx]
            
            if self.transform:
                mix_img = self.transform(mix_img)
            
            lam = np.random.beta(self.mixup_alpha, self.mixup_alpha)
            img = lam * img + (1 - lam) * mix_img
            
            # Return soft label
            label_onehot = torch.zeros(self.num_classes)
            label_onehot[label] = lam
            label_onehot[mix_label] += (1 - lam)
            return img, label_onehot
        
        return img, label

print('Dataset class defined')

In [ ]:
# Advanced transforms
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(224),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3, hue=0.1),
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),
    transforms.RandomGrayscale(p=0.05),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.2))
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load datasets
dataset_path = '/kaggle/input/datasets/ricardominor/vehicle-data/mexican_market'
print('Loading datasets...')
train_ds = VehicleDataset(f'{dataset_path}/train', train_transform, use_mixup=True)
val_ds = VehicleDataset(f'{dataset_path}/val', val_transform, use_mixup=False)
print(f'Train: {len(train_ds)} images, {train_ds.num_classes} classes')
print(f'Val: {len(val_ds)} images')

In [ ]:
# Model - MobileNetV3-Large (bigger, more accurate)
print('Creating model...')

model = models.mobilenet_v3_large(pretrained=True)

# Replace classifier with better architecture
num_features = model.classifier[0].in_features
model.classifier = nn.Sequential(
    nn.Linear(num_features, 1024),
    nn.Hardswish(),
    nn.Dropout(p=0.3),
    nn.Linear(1024, 512),
    nn.Hardswish(),
    nn.Dropout(p=0.2),
    nn.Linear(512, train_ds.num_classes)
)

model = model.to(device)

# DataLoaders
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

# Loss with label smoothing
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

# Optimizer with weight decay
optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

# Cosine annealing scheduler
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)

print(f'Model: MobileNetV3-Large')
print(f'Classes: {train_ds.num_classes}')
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Training with mixed precision
from torch.cuda.amp import autocast, GradScaler

EPOCHS = 50
best_acc = 0.0
patience = 10
patience_counter = 0

scaler = GradScaler()

print(f'Training for {EPOCHS} epochs with mixed precision...\n')
start_time = time.time()

for epoch in range(EPOCHS):
    # Train
    model.train()
    train_loss = 0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}')
    for images, labels in pbar:
        images = images.to(device)
        
        # Handle mixup (soft labels) vs regular labels
        if isinstance(labels, torch.Tensor) and labels.dim() == 2:
            labels = labels.to(device)
            use_soft = True
        else:
            labels = labels.to(device)
            use_soft = False
        
        optimizer.zero_grad()
        
        with autocast():
            outputs = model(images)
            if use_soft:
                loss = -(labels * torch.log_softmax(outputs, dim=1)).sum(dim=1).mean()
            else:
                loss = criterion(outputs, labels)
        
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        
        train_loss += loss.item()
        _, predicted = outputs.max(1)
        if use_soft:
            _, true_labels = labels.max(1)
            total += true_labels.size(0)
            correct += predicted.eq(true_labels).sum().item()
        else:
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100.*correct/total:.1f}%'})
    
    train_acc = 100. * correct / total
    scheduler.step()
    
    # Validate
    model.eval()
    val_loss = 0
    val_correct = 0
    val_total = 0
    
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
    
    val_acc = 100. * val_correct / val_total
    
    print(f'Epoch {epoch+1}: Train={train_acc:.1f}% Val={val_acc:.1f}% LR={optimizer.param_groups[0]["lr"]:.6f}')
    
    # Save best model
    if val_acc > best_acc:
        best_acc = val_acc
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'val_acc': val_acc,
            'num_classes': train_ds.num_classes,
            'class_to_idx': train_ds.class_to_idx,
            'idx_to_class': train_ds.idx_to_class,
        }, 'best_model.pth')
        print(f'  ✓ Saved (acc: {val_acc:.1f}%)')
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f'\nEarly stopping at epoch {epoch+1}')
            break

elapsed = time.time() - start_time
print(f'\n{"="*50}')
print(f'Training complete!')
print(f'Best accuracy: {best_acc:.1f}%')
print(f'Time: {elapsed/60:.1f} minutes')
print(f'{"="*50}')

In [ ]:
# Test-Time Augmentation (TTA) for better accuracy
print('Testing with TTA...\n')

checkpoint = torch.load('best_model.pth', map_location='cpu')
model = models.mobilenet_v3_large(pretrained=False)
num_features = model.classifier[0].in_features
model.classifier = nn.Sequential(
    nn.Linear(num_features, 1024),
    nn.Hardswish(),
    nn.Dropout(p=0.3),
    nn.Linear(1024, 512),
    nn.Hardswish(),
    nn.Dropout(p=0.2),
    nn.Linear(512, checkpoint['num_classes'])
)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()

idx_to_class = checkpoint['idx_to_class']

# TTA transforms
tta_transforms = [
    transforms.Compose([transforms.Resize((224, 224)), transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
    transforms.Compose([transforms.Resize((256, 256)), transforms.CenterCrop(224), transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
    transforms.Compose([transforms.Resize((256, 256)), transforms.RandomCrop(224), transforms.RandomHorizontalFlip(p=1.0), transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])]),
]

# Test on validation set
correct_standard = 0
correct_tta = 0
total = 0

for img_path in tqdm(val_ds.images, desc='TTA Testing'):
    img = Image.open(img_path).convert('RGB')
    actual = val_ds.labels[total]
    
    # Standard prediction
    inp = val_transform(img).unsqueeze(0).to(device)
    with torch.no_grad():
        out = model(inp)
        _, pred = out.max(1)
    if pred.item() == actual:
        correct_standard += 1
    
    # TTA prediction
    tta_outputs = []
    for t in tta_transforms:
        inp = t(img).unsqueeze(0).to(device)
        with torch.no_grad():
            out = model(inp)
            tta_outputs.append(out)
    
    avg_output = torch.stack(tta_outputs).mean(0)
    _, tta_pred = avg_output.max(1)
    if tta_pred.item() == actual:
        correct_tta += 1
    
    total += 1

print(f'\nStandard Accuracy: {correct_standard/total*100:.1f}%')
print(f'TTA Accuracy: {correct_tta/total*100:.1f}%')
print(f'Improvement: +{(correct_tta-correct_standard)/total*100:.1f}%')

In [ ]:
# Export model
!pip install -q onnx

import torch.onnx
import json

checkpoint = torch.load('best_model.pth', map_location='cpu')

# Export ONNX
dummy = torch.randn(1, 3, 224, 224)
torch.onnx.export(model.cpu(), dummy, 'vehicle_classifier.onnx',
    export_params=True, opset_version=11,
    input_names=['input'], output_names=['output'])
print('ONNX exported')

# Save labels
labels = {str(k): v for k, v in checkpoint['idx_to_class'].items()}
with open('vehicle_labels.json', 'w') as f:
    json.dump(labels, f, indent=2)
print('Labels saved')

# Show model size
import os
for f in ['best_model.pth', 'vehicle_classifier.onnx', 'vehicle_labels.json']:
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024 / 1024
        print(f'{f}: {size:.1f} MB')

print('\nDone! Download all 3 files.')